## Writing Tool Schemas for Claude

# Unit 21: Tool Integration, Semantic Interfaces, and Function Schemas

## Introduction

Welcome to your first lesson in developing Claude agents with native tool integration! In this module, you will learn the foundational skill of designing declarative **Function Schemas**. These schemas enable Claude to safely understand, select, and request the execution of your custom backend capabilities through a process known as **Function Calling**.

By the end of this lesson, you will understand how to write type-hinted Python functions and engineer corresponding JSON schemas that describe these execution interfaces to Claude. These schemas serve as the critical abstraction bridge allowing Claude to understand exactly what your functions do and how to structure arguments for them—even though **Claude never sees your actual source code**. Mastering this decoupling is essential before you can deploy a complete, autonomous agent loop capable of executing tools and reasoning over their outputs.

---

## The Function Calling Lifecycle

Function calling is the mechanism that transforms Claude from an isolated text generator into an embodied agent capable of interacting with external databases, APIs, and local runtimes.

The complete agent-tool execution loop follows a strict five-stage transaction cycle:

```text
 ┌──────────────────────────┐        1. Ingests Schemas        ┌──────────────────────────┐
 │                          │ ───────────────────────────────► │                          │
 │      Client Runtime      │                                  │       Claude Model       │
 │   (Local Python Script)  │ ◄─────────────────────────────── │     (Inference Engine)   │
 │                          │    3. Returns tool_use Token     │                          │
 └──────────────────────────┘                                  └──────────────────────────┘
      │                ▲
      │ 4. Executes    │ 5. Appends
      │    Function    │    Result
      ▼                │
 ┌──────────────────────────┐
 │   Internal Tool Space    │
 │     (functions.py)       │
 └──────────────────────────┘

```

1. **Schema Provisioning:** When dispatching a user message to the Anthropic API, your local client runtime includes an array of structural JSON descriptions outlining available capabilities.
2. **Semantic Request Analysis:** Claude reads the incoming user prompt against the provided schemas to determine if any of the declared tools are necessary to satisfy the prompt's intent.
3. **The Tool Request Token:** If Claude decides a tool is required, it halts text generation and returns a structured `tool_use` message payload specifying the targeted function name and an object containing explicitly mapped arguments.
4. **Local Subprocess Execution:** Your local runtime catches Claude's `tool_use` token, stops the inference loop, intercepts the arguments payload, looks up the function within an internal execution mapping table, and runs the local Python code.
5. **Context Ingestion:** The runtime captures the value returned by your Python code, wraps it inside a `tool_result` message envelope, and passes it back to Claude. Claude reviews the result to form its final natural language answer or determine if it needs to trigger additional tools.

The key insight is that **Claude only interacts with your JSON schemas, never your executable code blocks.** This strict separation of concerns gives you total architectural freedom to structure your backend Python files however you like; Claude makes its operational decisions based entirely on the semantic quality of your schema descriptions.

---

## Writing Clear Python Tool Functions

When building tools for an agent ecosystem, your local Python code serves a dual purpose: it implements the actual logical operations and provides a structural reference for drafting your JSON declarations.

Create a file named `functions.py` to hold your standalone execution tools:

```python
def sum_numbers(a: float, b: float) -> float:
    """
    Sum two numbers and return the result.
        
    Args:
        a (float): First number to add
        b (float): Second number to add
            
    Returns:
        float: The sum of a and b
    """
    return a + b

def multiply_numbers(a: float, b: float) -> float:
    """
    Multiply two numbers and return the result.
        
    Args:
        a (float): First number to multiply
        b (float): Second number to multiply
            
    Returns:
        float: The product of a and b
    """
    return a * b

```

### Writing Idiomatic Tool Implementations:

* **Strict Type Hinting:** Enforcing explicit input types (`a: float`) and clear return definitions (`-> float`) anchors your parameters, preventing the ingestion of ambiguous data structures.
* **Explanatory Docstrings:** Writing thorough descriptions of a function's behavior, argument matrices, and return fields establishes a clear blueprint for translating that code into a schema that Claude can reason over.

---

## Engineering JSON Schemas for Tool Declaration

JSON schemas are the structured manifests that tell Claude exactly what a tool does and how to form its inputs. Because this is the only data Claude receives about your application capabilities, the clarity of your descriptions directly dictates the routing accuracy of the agent.

Create a file named `schemas.json` to collect your declarations:

```json
[
  {
    "name": "sum_numbers",
    "description": "Sum two numbers and return the result",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number to add"
        },
        "b": {
          "type": "number",
          "description": "Second number to add"
        }
      },
      "required": ["a", "b"]
    }
  },
  {
    "name": "multiply_numbers",
    "description": "Multiply two numbers and return the result",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number to multiply"
        },
        "b": {
          "type": "number",
          "description": "Second number to multiply"
        }
      },
      "required": ["a", "b"]
    }
  }
]

```

### Deconstructing the Schema Specification:

* **`name`:** A clear string identifier used to index the tool. While it does not technically have to match your Python function name, keeping them identical makes your code significantly cleaner.
* **`description`:** The most critical semantic variable in the block. Claude uses this field to evaluate user intent. If your description is ambiguous or poorly defined, Claude may misroute tasks or ignore the tool entirely.
* **`input_schema`:** Defines parameters using the standardized JSON Schema specification format. It declares the wrapper object type, establishes specific validation properties, and lists expected data types (e.g., mapping Python's `float` to JSON's `"type": "number"`).
* **`required`:** An array of parameter keys that are mandatory for execution. This tells Claude it must extract these fields before dispatching a `tool_use` request.

---

## Constructing the Inversion Execution Map

Once your functions are isolated inside `functions.py` and your schemas are organized within `schemas.json`, you must build a coordination bridge. This layout decouples the declarations passed to Claude from your internal function execution references.

Create a controller file named `main.py` to wire these spaces together:

```python
import json
from functions import sum_numbers, multiply_numbers

# 1. Ingest the external schemas manifest to pass to the Anthropic API
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# 2. Build the internal routing map linking schema tokens to Python references
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

```

> ⚠️ **Critical Alignment Invariant:** The dictionary string keys configured inside your `tools` lookup map must **exactly match** the values declared in the `name` fields of your `schemas.json` file. Any naming mismatch will cause your client to throw a runtime error when attempting to look up and execute code requested by Claude.

---

## Verifying Integrity and Testing Execution Maps

Before registering your schemas with an active Anthropic API client loop, verify the structural integrity of your JSON ingestion and ensure your function mapping dictionary resolves calls correctly.

Append these diagnostic print blocks to the bottom of your `main.py` script:

```python
# Verify that the JSON schemas are properly formatted and readable
print("=== VERIFYING ACCESSIBLE SCHEMAS ===")
print(json.dumps(tool_schemas, indent=2))

print("\n=== TESTING FUNCTION ROUTING LIFECYCLE ===")
# Simulate a local lookup execution pass for 'sum_numbers'
result_sum = tools['sum_numbers'](10, 5)
print(f"Lookup Execution -> tools['sum_numbers'](10, 5) = {result_sum}")

# Simulate a local lookup execution pass for 'multiply_numbers'
result_mult = tools['multiply_numbers'](4, 7)
print(f"Lookup Execution -> tools['multiply_numbers'](4, 7) = {result_mult}")

```

Executing `python main.py` in your terminal environment outputs clean tracking logs:

```text
=== VERIFYING ACCESSIBLE SCHEMAS ===
[
  {
    "name": "sum_numbers",
    "description": "Sum two numbers and return the result",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number to add"
        },
        "b": {
          "type": "number",
          "description": "Second number to add"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  },
  {
    "name": "multiply_numbers",
    "description": "Multiply two numbers and return the result",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number to multiply"
        },
        "b": {
          "type": "number",
          "description": "Second number to multiply"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
]

=== TESTING FUNCTION ROUTING LIFECYCLE ===
Lookup Execution -> tools['sum_numbers'](10, 5) = 15
Lookup Execution -> tools['multiply_numbers'](4, 7) = 28

```

This output confirms that your application can cleanly load schemas from disk and look up and run target functions using string lookups.

---

## Summary & Next Steps

You have successfully implemented the core setup required to prepare custom tool integrations for Claude Code agents. You now understand how to build type-hinted Python functions, express capabilities through strict JSON schemas, and implement an internal lookup map to route string-based tool requests back to native code.

Keep the separation of concerns in mind: **Claude manages semantic tool routing using your schema summaries, while your local application handles actual runtime execution.** The quality and precision of your schema descriptions directly govern the intelligence and reliability of your agent workflows.

Let's head into the practical lab exercises to build out and register your first custom tool schemas!

## Building a Well Documented Tool Function

Now that you understand how Claude uses tool schemas to interact with functions, it's time to build your first tool function from scratch! You'll create a well-documented Python function that follows the patterns shown in the lesson, then test it to make sure everything works correctly.

In this task, you will:

    Write a complete sum_numbers function in the functions.py file that:
        Accepts two parameters, a and b, both with float type hints
        Returns a float result with the proper return type hint
        Includes a detailed docstring that explains the function's purpose, describes each parameter in an Args section, and specifies the return value in a Returns section
        Implements the logic to return the sum of the two input numbers

    After creating your function, switch to main.py and:
        Import your sum_numbers function
        Call it with the values 8 and 12
        Print the result to verify that your function works as expected

This exercise gives you hands-on practice with the foundational skills you'll need when building Claude agents with custom tools!

```python

/usercode/FILESYSTEM$ cat functions.py 
# TODO: Write the sum_numbers function with proper type hints, docstring, and implementation

/usercode/FILESYSTEM$ cat main.py 
# TODO: Import the sum_numbers function from functions

# TODO: Call sum_numbers with values 8 and 12, store the result in a variable

# TODO: Print the result/usercode/FILESYSTEM$ 

```

To build a reliable tool ecosystem for a Claude agent, you must ensure your Python functions are structurally sound, strictly typed, and thoroughly documented. Even though Claude only sees the schema, establishing a robust source definition ensures clean translation mapping and prevents input data corruption.

Here is the complete implementation for both of your file blocks:

### 1. The Tool Space (`functions.py`)

Open `functions.py` and implement your type-hinted function and detailed structural docstring:

```python
def sum_numbers(a: float, b: float) -> float:
    """
    Sum two numbers and return the result.
        
    Args:
        a (float): First number to add
        b (float): Second number to add
            
    Returns:
        float: The sum of a and b
    """
    return a + b

```

---

### 2. The Main Runtime (`main.py`)

Open `main.py` to import your new tool function, invoke it with the test boundaries, and log the execution state:

```python
# Import the sum_numbers function from functions
from functions import sum_numbers

# Call sum_numbers with values 8 and 12, store the result in a variable
result = sum_numbers(8.0, 12.0)

# Print the result to verify the execution lifecycle
print(f"Result of calling sum_numbers(8, 12): {result}")

```

---

### Why This Design Pattern is Essential:

* **Interface Contract Enforcement:** Adding strict typing constraints (`a: float`) ensures that any upstream data parsing logic in your agent application handles string-to-float formatting accurately before invoking the backend logic.
* **Decoupled Architecture:** This separation structures `functions.py` purely as a declarative inventory of tools, while `main.py` functions as the execution orchestrator—the exact layout you will need when expanding this setup to load automated JSON schemas!

## Writing a Tool Schema for Claude

## Working with Multiple Tools

## Mapping Schema Tools to Functions

## Mapping Schema Tools to Functions